
# 🔧 Gemini Tool Selection — See How an LLM Chooses a Tool

### Dinesh AI Academy | Day 3 — Tools & Workflows

**Learning objective:**  
Understand the technical process behind tool selection without pretending to see the model's private chain-of-thought.

We will make the model choose between:
- `calculate` → mathematical calculations
- `get_weather` → weather information
- `send_email` → email sending

### The key idea

```text
User Request
     ↓
LLM receives:
  • User message
  • Conversation context
  • Available tool definitions
     ↓
Model generates either:
  • Normal text response
  • Structured function/tool call
     ↓
Our Python application executes the requested tool
     ↓
Tool result goes back to the LLM
     ↓
Final response
```

> **Important:** We can observe the tool declaration and the function call generated by Gemini. We cannot see Gemini's private internal reasoning or hidden chain-of-thought.



## 1. Install the Gemini SDK

We use Google's current `google-genai` Python SDK.


In [1]:
!pip -q install -U google-genai


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("✅ API key loaded successfully.")

✅ API key loaded successfully.



## 2. Add your Gemini API key

For Colab, enter the key when prompted. Do not hard-code or share your API key.


In [7]:

from google import genai
from getpass import getpass

client = genai.Client(api_key=GAISTUDIO_API_KEY)

MODEL = "gemini-3.6-flash"

print("Gemini client ready.")


Gemini client ready.



## 3. First: Give Gemini some tools

The LLM cannot directly execute our Python functions.

Instead, we describe the capabilities that are available to it.

Think of this as giving the model a **menu of capabilities**.

| Tool | Capability |
|---|---|
| `calculate` | Performs mathematical calculations |
| `get_weather` | Gets weather information for a city |
| `send_email` | Sends an email |

The descriptions matter because they help the model understand **when a tool can satisfy the user's request**.


In [8]:

from google.genai import types

calculate_declaration = types.FunctionDeclaration(
    name="calculate",
    description="Perform a mathematical calculation using two numbers.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "a": types.Schema(type=types.Type.NUMBER, description="First number"),
            "b": types.Schema(type=types.Type.NUMBER, description="Second number"),
            "operation": types.Schema(
                type=types.Type.STRING,
                description="Operation such as add, subtract, multiply, or divide"
            ),
        },
        required=["a", "b", "operation"],
    ),
)

weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current weather information for a city.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "city": types.Schema(type=types.Type.STRING, description="City name"),
        },
        required=["city"],
    ),
)

email_declaration = types.FunctionDeclaration(
    name="send_email",
    description="Send an email to a recipient with a subject and message.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "recipient": types.Schema(type=types.Type.STRING, description="Email recipient"),
            "subject": types.Schema(type=types.Type.STRING, description="Email subject"),
            "message": types.Schema(type=types.Type.STRING, description="Email body"),
        },
        required=["recipient", "subject", "message"],
    ),
)

tool = types.Tool(
    function_declarations=[
        calculate_declaration,
        weather_declaration,
        email_declaration,
    ]
)

config = types.GenerateContentConfig(
    tools=[tool],
    temperature=0
)

print("Available tools:")
for name, description in [
    ("calculate", "Perform mathematical calculations"),
    ("get_weather", "Get weather information for a city"),
    ("send_email", "Send an email"),
]:
    print(f"  • {name:15} → {description}")


Available tools:
  • calculate       → Perform mathematical calculations
  • get_weather     → Get weather information for a city
  • send_email      → Send an email



## 4. The important experiment — ask for a calculation

We are going to inspect the **actual structured output** returned by Gemini.

Question:

> What is 25 × 40?

We expect the model to request the `calculate` tool.

Notice the important distinction:

**Gemini does NOT run our Python calculator.**  
Gemini generates a request asking our application to call it.


In [9]:

user_prompt = "What is 25 × 40?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config
)

print("USER REQUEST:")
print(user_prompt)

print("\nRAW RESPONSE OBJECT:")
print(response)

print("\nRESPONSE PARTS:")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"\nPart {i}:")
    print("Text:", part.text)
    print("Function call:", part.function_call)


USER REQUEST:
What is 25 × 40?

RAW RESPONSE OBJECT:
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'a': 25,
            'b': 40,
            'operation': 'multiply'
          },
          id='call_151860',
          name='calculate'
        ),
        thought_signature=b'\x12\x9c\x04\n\x99\x04\x01\x11M2\x0f\xb5]R\x90\xa9X\x9d-\xaf\xffe1r?\xb2\xf3\x94\xa5\xc5\xd6\x10\xf2\x8e\x85\x03,d\xfa\xee.\xcf\x07X_\xc9\x84~\x89t\\kM\xc9\xbc\xa7\x17\x93\x1f?\xf4c\x8b\xceZ\xa5\x89\xec\x8a\xe7\x9f\x05\x7f\xac6}\x18\xe4\xa0\xd7\x87\xac\x02\tw\xb5\xa4\xe2\xe4\xe1\xd0\xc7X\xd5o#_...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.6-flash' prompt_feedback=None response_id='sR-qauSmKvOqg8UPnLCc4QQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=24


## 5. Extract the tool call

Now let's inspect only the useful fields.

A typical result will look conceptually like:

```text
Function name : calculate
Arguments     : {
    "a": 25,
    "b": 40,
    "operation": "multiply"
}
```

This is the **observable tool-selection decision** produced by the model.

We are not saying:

> "The model secretly calculated a score of 0.92 for calculator."

That internal process is not exposed.

Instead, we can accurately say:

> **The model received the user's request and the available tool definitions, then generated a structured function call for `calculate` with the required arguments.**


In [10]:

function_call = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        function_call = part.function_call
        break

if function_call:
    print("========== MODEL TOOL SELECTION ==========")
    print("Selected tool :", function_call.name)
    print("Arguments     :", dict(function_call.args))
else:
    print("No tool was selected.")
    print("Model response:", response.text)


========== MODEL TOOL SELECTION ==========
Selected tool : calculate
Arguments     : {'a': 25, 'operation': 'multiply', 'b': 40}



## 6. Visualize what actually happened

```text
┌──────────────────────────┐
│ User                     │
│ "What is 25 × 40?"       │
└────────────┬─────────────┘
             │
             ▼
┌────────────────────────────────────────┐
│ Gemini receives                        │
│                                        │
│ User request                           │
│ +                                      │
│ Tool: calculate                        │
│ Tool: get_weather                      │
│ Tool: send_email                       │
└──────────────────┬─────────────────────┘
                   │
                   ▼
            ┌─────────────┐
            │ Gemini LLM  │
            └──────┬──────┘
                   │
                   │ structured output
                   ▼
       ┌─────────────────────────┐
       │ function_call           │
       │ name = calculate        │
       │ a = 25                  │
       │ b = 40                  │
       │ operation = multiply    │
       └────────────┬────────────┘
                    │
                    ▼
          ┌────────────────────┐
          │ Python Application │
          │ executes function  │
          └────────────────────┘
```



## 7. Let our application execute the selected tool

This is where the application becomes important.

The model requested:

```text
calculate(25, 40, "multiply")
```

Our Python program executes the real function.


In [11]:

def calculate(a, b, operation):
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            return "Cannot divide by zero"
        return a / b
    else:
        return f"Unsupported operation: {operation}"


def get_weather(city):
    # Demo only — replace with a real weather API in production.
    return f"Demo weather result for {city}: 28°C, partly cloudy"


def send_email(recipient, subject, message):
    # Demo only — do NOT actually send an email in this notebook.
    return f"Demo email result: email would be sent to {recipient}"


In [12]:

if function_call:
    name = function_call.name
    args = dict(function_call.args)

    if name == "calculate":
        tool_result = calculate(**args)
    elif name == "get_weather":
        tool_result = get_weather(**args)
    elif name == "send_email":
        tool_result = send_email(**args)
    else:
        tool_result = f"Unknown tool: {name}"

    print("TOOL EXECUTION")
    print("Tool   :", name)
    print("Args   :", args)
    print("Result :", tool_result)


TOOL EXECUTION
Tool   : calculate
Args   : {'a': 25, 'operation': 'multiply', 'b': 40}
Result : 1000



## 8. Send the tool result back to Gemini

The loop is not finished after tool execution.

The application sends the tool result back to the model so Gemini can turn the machine-readable result into a natural-language answer.


In [13]:

if function_call:
    tool_response_part = types.Part.from_function_response(
        name=function_call.name,
        response={"result": tool_result},
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=user_prompt)]
            ),
            response.candidates[0].content,
            types.Content(
                role="tool",
                parts=[tool_response_part]
            ),
        ],
        config=config
    )

    print("FINAL LLM RESPONSE:")
    print(final_response.text)


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': "Role 'tool' is not supported. Please use a valid role: SYSTEM, SYSTEM_1, USER, ASSISTANT, DEVELOPER, CONTEXT, USER_CONTEXT, MODEL, USER.", 'status': 'INVALID_ARGUMENT'}}


# 9. Now test different user intents

This is the most useful classroom experiment.

We keep the **same three tools** and change only the user's request.

Try:

1. `What is 125 * 40?`
2. `What is the weather in Delhi?`
3. `Send an email to Rahul saying the meeting is at 5 PM.`
4. `Explain what RAG is.`
5. `What is the weather in Mumbai and what is 20 * 5?`

Ask students:

> **What changed — the tools or the user's goal?**

The tools stay the same. The user goal changes, and the model generates a different structured action.


In [ ]:

def inspect_tool_selection(user_prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config=config
    )

    print("=" * 70)
    print("USER:", user_prompt)
    print("-" * 70)

    found = False

    for part in response.candidates[0].content.parts:
        if part.function_call:
            found = True
            print("SELECTED TOOL :", part.function_call.name)
            print("ARGUMENTS      :", dict(part.function_call.args))

    if not found:
        print("SELECTED TOOL : None")
        print("MODEL TEXT    :", response.text)

    return response


test_prompts = [
    "What is 125 * 40?",
    "What is the weather in Delhi?",
    "Send an email to Rahul saying the meeting is at 5 PM.",
    "Explain what RAG is.",
]

for prompt in test_prompts:
    inspect_tool_selection(prompt)



## 10. The key technical insight: tool descriptions influence selection

Consider these two descriptions:

### Tool A

```text
calculate:
"Perform a mathematical calculation using two numbers."
```

### Tool B

```text
get_weather:
"Get the current weather information for a city."
```

For:

```text
"What is 25 × 40?"
```

the model can map the user's intent to the capability described by `calculate`.

For:

```text
"What is the weather in Delhi?"
```

the model can map the user's intent to `get_weather`.

So the architecture is:

```text
USER INTENT
     +
TOOL DESCRIPTIONS
     +
CONVERSATION CONTEXT
     ↓
LLM MODEL COMPUTATION
     ↓
STRUCTURED TOOL CALL
     ↓
APPLICATION EXECUTES TOOL
```

**Do not teach this as a hard-coded `if/else` inside Gemini.**

Your application may use `if/else` to execute the function after Gemini has requested it. The model's tool selection is generated by the model based on the request and the supplied tool definitions.



# 11. A more technical mental model

Students often ask:

> "How exactly does the LLM know which tool to select?"

A useful engineering explanation is:

### Step 1 — We expose capabilities

```text
calculate
get_weather
send_email
```

with names, descriptions, parameters, and schemas.

### Step 2 — The user expresses a goal

```text
"How much is 500 × 20?"
```

### Step 3 — The model processes the request together with the available tool definitions

The model determines that a calculator capability is relevant.

### Step 4 — The model generates structured output

```json
{
  "name": "calculate",
  "arguments": {
    "a": 500,
    "b": 20,
    "operation": "multiply"
  }
}
```

### Step 5 — The application validates and executes it

```python
calculate(500, 20, "multiply")
```

### Step 6 — The result is returned to the model

```json
{
  "result": 10000
}
```

### Step 7 — The model generates the final answer

```text
500 × 20 = 10,000.
```

This is the boundary between **model reasoning/generation** and **application execution**.



# 12. Important: tool selection ≠ tool execution

This distinction is critical for AI engineering.

| Responsibility | Who does it? |
|---|---|
| Understand user's request | LLM |
| Decide/generate which tool call to request | LLM |
| Generate tool arguments | LLM |
| Validate permissions | Application |
| Execute Python/API/database function | Application |
| Return tool result | Application |
| Explain result to user | LLM |

### Security example

If the model requests:

```text
send_email(recipient="someone@example.com", ...)
```

that does **not** mean the email should automatically be sent.

A production application should consider:

```text
LLM Tool Request
      ↓
Validate schema
      ↓
Check permissions
      ↓
Apply business rules
      ↓
Optional human approval
      ↓
Execute
```

This is one reason tool calling is an **AI engineering** problem, not just an LLM prompting problem.



# 13. Challenge — make the selection harder

Add two more tools:

```text
search_web
search_documents
```

Then ask:

> "Find information about our company's leave policy."

Ask students:

1. Which tool should be selected?
2. What information in the tool descriptions helps the model?
3. What arguments should the model generate?
4. What should the application validate before executing the tool?

Then change the request:

> "Search the public web for the latest information about Python."

Observe how the appropriate capability can change while the available tools remain the same.

---

# 14. From Tool Calling → Workflows → Agents

This notebook demonstrates **tool calling**.

The progression for Day 3 and Day 4 is:

```text
TOOL
A capability the application can execute
        ↓
WORKFLOW
A predefined sequence of actions
        ↓
AGENT
The model can decide what action/tool to take next
        ↓
MULTI-STEP AGENT
Observe → Decide → Act → Observe → Decide → Act
```

### Simple example

```text
User:
"Find the price of a product and calculate the price after 10% discount."

        ↓

LLM
        ↓
search_product()
        ↓
product price = ₹2,000
        ↓
LLM
        ↓
calculate()
        ↓
discounted price = ₹1,800
        ↓
LLM
        ↓
Final answer
```

This is the bridge from today's **tool calling** lesson to tomorrow's **AI Agents** lesson.



# 🎯 Day 3 Takeaway

Students should leave this notebook understanding these six ideas:

1. **Tools give an LLM external capabilities.**
2. **Tool definitions tell the model what capabilities are available.**
3. **The model can generate a structured function/tool call.**
4. **The application executes the requested function.**
5. **The tool result is sent back to the model for the final response.**
6. **We can observe the generated tool call, but not the model's private chain-of-thought.**

### One sentence to remember

> **The LLM decides what tool call to request; the application decides whether and how that tool call is actually executed.**

### Next step

**Tools → Workflows → Agents → MCP**



## Official references

- Gemini Function Calling: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API Quickstart: https://ai.google.dev/gemini-api/docs/get-started
- Gemini Tools: https://ai.google.dev/gemini-api/docs/tools
- Google AI Studio: https://aistudio.google.com/
